# 06 — Serving

## About

**Purpose:** Package the XGBoost exclusion-risk model behind a FastAPI service, smoke-test it locally, and containerise it for GCP Cloud Run.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-05-16<br>
**Notes:** Loads `model.ubj` directly via xgboost (no MLflow at serve time — slimmer container). Target-encoding maps come from notebook 03's `serving/encoding_maps.json`. The local Uvicorn server stays alive in the kernel; re-running the run-server cell restarts it. Docker build cell requires Docker Desktop running.<br>
**Description:** Locates the MLflow-logged model artifact, copies it into `serving/`, writes `app.py` / `requirements.txt` / `Dockerfile`, boots a local Uvicorn server on port 8000, smoke-tests `/health` and `/predict` via httpx, then builds the production Docker image tagged `healthcare-exclusion-risk:latest` ready for push to Artifact Registry.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-05-16 | 1.0     | Ganapathy K | Initial version |

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Setup
### 1.1 Imports

In [ ]:
import warnings
import json
import time
import shutil
import logging
import subprocess
from datetime import datetime
from pathlib import Path

import pandas as pd
import xgboost as xgb
import httpx

warnings.filterwarnings("ignore")

### 1.2 Configure logging

In [ ]:
log_folder = Path("logs")
log_folder.mkdir(exist_ok=True)
log_filename = log_folder / f"run_{datetime.now().strftime('%Y-%m-%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(log_filename, encoding='utf-8'),
        logging.StreamHandler(),
    ],
    force=True,
)
logger = logging.getLogger(__name__)

### 1.3 Config

In [ ]:
mlflow_model_artifact_path = Path("../notebooks/mlruns/0/models/m-787607a155fb46b29bc9fcd13f990822/artifacts/model.ubj")

serving_dir = Path("../serving")
serving_model_path = serving_dir / "model.ubj"

local_server_host = "0.0.0.0"
local_server_port = 8000
local_server_url = f"http://localhost:{local_server_port}"

docker_image_tag = "healthcare-exclusion-risk:latest" 

## 2. Install Serving Dependencies

In [ ]:
subprocess.run(["pip", "install", "fastapi", "uvicorn[standard]", "httpx"], check=True)
logger.info("Serving dependencies installed: fastapi, uvicorn[standard], httpx")

## 3. Load and Verify Model

In [ ]:
logger.info(f"Model file exists: {mlflow_model_artifact_path.exists()}")
logger.info(f"Model path: {mlflow_model_artifact_path.resolve()}")

model = xgb.XGBClassifier()
model.load_model(mlflow_model_artifact_path)

logger.info(f"Features expected by model ({len(model.feature_names_in_)}):")
for feature in model.feature_names_in_:
    logger.info(f"  - {feature}")

## 4. Prepare serving/ Folder

In [ ]:
serving_dir.mkdir(exist_ok=True)
shutil.copy(mlflow_model_artifact_path, serving_model_path)

logger.info(f"serving/ ready at: {serving_dir.resolve()}")
logger.info(f"model.ubj copied to: {serving_model_path}")

## 5. Write app.py

In [ ]:
%%writefile ../serving/app.py
from fastapi import FastAPI
from pydantic import BaseModel
import xgboost as xgb
import pandas as pd
import json
from pathlib import Path

app = FastAPI(title="Healthcare Provider Exclusion Risk API")

MODEL_PATH = Path(__file__).parent / "model.ubj"
model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

ENCODING_MAPS_PATH = Path(__file__).parent / "encoding_maps.json"
with open(ENCODING_MAPS_PATH) as file:
    encoding_maps = json.load(file)

FEATURE_COLUMNS = [
    "Entity Type Code",
    "Provider Business Mailing Address State Name",
    "Provider Business Mailing Address Telephone Number",
    "Provider Business Practice Location Address State Name",
    "Healthcare Provider Taxonomy Code_1",
    "Provider License Number State Code_1",
    "Provider Enumeration Year",
    "Last Update Year",
    "Provider Sex Code_F",
    "Provider Sex Code_M",
    "Provider Sex Code_U",
    "Healthcare Provider Primary Taxonomy Switch_1_N",
    "Healthcare Provider Primary Taxonomy Switch_1_Y",
    "Is Sole Proprietor_N",
    "Is Sole Proprietor_X",
    "Is Sole Proprietor_Y",
]


class ProviderInput(BaseModel):
    Entity_Type_Code: int
    Provider_Business_Mailing_Address_State_Name: str
    Provider_Business_Mailing_Address_Telephone_Number: float
    Provider_Business_Practice_Location_Address_State_Name: str
    Healthcare_Provider_Taxonomy_Code_1: str
    Provider_License_Number_State_Code_1: str
    Provider_Enumeration_Year: int
    Last_Update_Year: int
    Provider_Sex_Code: str
    Healthcare_Provider_Primary_Taxonomy_Switch_1: str
    Is_Sole_Proprietor: str


def encode_input(data: ProviderInput) -> pd.DataFrame:
    taxonomy_mean = encoding_maps["Healthcare Provider Taxonomy Code_1"].get(data.Healthcare_Provider_Taxonomy_Code_1, 0.0)
    mailing_state_mean = encoding_maps["Provider Business Mailing Address State Name"].get(data.Provider_Business_Mailing_Address_State_Name, 0.0)
    practice_state_mean = encoding_maps["Provider Business Practice Location Address State Name"].get(data.Provider_Business_Practice_Location_Address_State_Name, 0.0)
    license_state_mean = encoding_maps["Provider License Number State Code_1"].get(data.Provider_License_Number_State_Code_1, 0.0)

    row = {
        "Entity Type Code": data.Entity_Type_Code,
        "Provider Business Mailing Address State Name": mailing_state_mean,
        "Provider Business Mailing Address Telephone Number": data.Provider_Business_Mailing_Address_Telephone_Number,
        "Provider Business Practice Location Address State Name": practice_state_mean,
        "Healthcare Provider Taxonomy Code_1": taxonomy_mean,
        "Provider License Number State Code_1": license_state_mean,
        "Provider Enumeration Year": data.Provider_Enumeration_Year,
        "Last Update Year": data.Last_Update_Year,
        "Provider Sex Code_F": int(data.Provider_Sex_Code == "F"),
        "Provider Sex Code_M": int(data.Provider_Sex_Code == "M"),
        "Provider Sex Code_U": int(data.Provider_Sex_Code == "U"),
        "Healthcare Provider Primary Taxonomy Switch_1_N": int(data.Healthcare_Provider_Primary_Taxonomy_Switch_1 == "N"),
        "Healthcare Provider Primary Taxonomy Switch_1_Y": int(data.Healthcare_Provider_Primary_Taxonomy_Switch_1 == "Y"),
        "Is Sole Proprietor_N": int(data.Is_Sole_Proprietor == "N"),
        "Is Sole Proprietor_X": int(data.Is_Sole_Proprietor == "X"),
        "Is Sole Proprietor_Y": int(data.Is_Sole_Proprietor == "Y"),
    }
    return pd.DataFrame([row], columns=FEATURE_COLUMNS)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(provider: ProviderInput):
    input_dataframe = encode_input(provider)
    exclusion_probability = float(model.predict_proba(input_dataframe)[0][1])
    prediction = int(model.predict(input_dataframe)[0])
    return {
        "excluded": bool(prediction),
        "exclusion_probability": round(exclusion_probability, 4),
        "risk_tier": "high" if exclusion_probability >= 0.5 else "low",
    }


## 6. Run Local Server

In [ ]:
try:
    server_process.terminate()
    time.sleep(1)
except NameError:
    pass

server_process = subprocess.Popen(
    ["uvicorn", "app:app", "--host", local_server_host, "--port", str(local_server_port)],
    cwd=str(serving_dir),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(3)
logger.info(f"Server restarted — PID: {server_process.pid} | URL: {local_server_url}")

## 7. Smoke Tests
### 7.1 Health check

In [ ]:
response = httpx.get(f"{local_server_url}/health")
print("Status:", response.status_code)
print("Response:", response.json())

### 7.2 Prediction

In [ ]:
sample_provider = {
    "Entity_Type_Code": 1,
    "Provider_Business_Mailing_Address_State_Name": "CA",
    "Provider_Business_Mailing_Address_Telephone_Number": 3105550100.0,
    "Provider_Business_Practice_Location_Address_State_Name": "CA",
    "Healthcare_Provider_Taxonomy_Code_1": "207Q00000X",
    "Provider_License_Number_State_Code_1": "CA",
    "Provider_Enumeration_Year": 2005,
    "Last_Update_Year": 2022,
    "Provider_Sex_Code": "M",
    "Healthcare_Provider_Primary_Taxonomy_Switch_1": "Y",
    "Is_Sole_Proprietor": "Y",
}

response = httpx.post(f"{local_server_url}/predict", json=sample_provider)
print("Status:", response.status_code)
print("Response:", response.json())

## 8. Write requirements.txt

In [ ]:
%%writefile ../serving/requirements.txt
fastapi
uvicorn[standard]
xgboost
pandas
scikit-learn

## 9. Write Dockerfile

In [ ]:
%%writefile ../serving/Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY model.ubj .
COPY encoding_maps.json .

EXPOSE 8080

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080"]

## 10. Build Docker Image

In [ ]:
result = subprocess.run(
    ["docker", "build", "-t", docker_image_tag, "."],
    cwd=str(serving_dir),
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)

## 11. Summary

**Pitch:** Packaged the XGBoost exclusion-risk model behind a FastAPI service, smoke-tested it locally, and containerised it for Cloud Run. Deployed live at `https://healthcare-exclusion-risk-445269150468.us-central1.run.app`.

### Approach
- Loaded `model.ubj` directly via xgboost (no MLflow at serve time — keeps the container slim)
- FastAPI app with `/health` (liveness) and `/predict` (encoded provider features → exclusion probability + risk tier)
- Target encoding maps from notebook 03 (`serving/encoding_maps.json`) loaded at server startup
- Local smoke tests via httpx against `localhost:8000`
- Docker image tagged `healthcare-exclusion-risk:latest` on `python:3.11-slim`, binds `0.0.0.0:8080` for Cloud Run

### Outputs
- `serving/app.py` — FastAPI service
- `serving/requirements.txt` — runtime deps (`fastapi`, `uvicorn[standard]`, `xgboost`, `pandas`, `scikit-learn`)
- `serving/Dockerfile` — container spec
- `serving/model.ubj` + `serving/encoding_maps.json` — runtime artifacts

### Deployed (2026-04-23)
- **Live URL:** `https://healthcare-exclusion-risk-445269150468.us-central1.run.app`
- **GCP Project:** `project-1352635a-58f8-4210-9b0`
- **Artifact Registry:** `us-central1-docker.pkg.dev/project-1352635a-58f8-4210-9b0/healthcare-repo/healthcare-exclusion-risk`
- Fix during deploy: `scikit-learn` was missing from requirements.txt — added before final build.

### Next iteration
- Currently serves XGBoost only. The LangGraph agent (notebook 05) and the RAG pipeline (notebook 04) are notebook-only prototypes. Once the Qdrant swap on 04/05 lands, deploy the agent as a second Cloud Run service alongside this one.